# 11 · Correlaciones: agenda por macrotema vs estándar externo (1980–2011)
**Proyecto PLN · Validación del corpus contra una fuente de verdad**

**Qué hace.** (1) Reconstruye la prevalencia anual por macrotema y por tópico desde `theta_documentos.parquet` con el mismo protocolo del notebook 08 de tu compañero (meses válidos, sin breves, igual peso por mes). (2) Construye el estándar externo (indicadores del Banco Mundial, eventos y calendario). (3) Contrasta ambos con un conjunto de **hipótesis congeladas** y entrega correlaciones, veredictos y gráficas.

**Archivos que asume ya cargados** (en la carpeta `RUTA`; el prefijo del nombre no importa):
`theta_documentos.parquet`, `macrocategorias.csv`, `meses_validos.csv`. Opcionales: `resumen_08.json` (toma de ahí los parámetros del protocolo), `prevalencias_ic.csv` (verificación contra el 08) y `data_estandar/secuestros_cnmh.csv` (serie del CNMH).

**Lectura rápida de los resultados.** Veredicto por hipótesis → *Soporta (fuerte / parcial)*, *Sin evidencia*, *Contradice*, *Sin potencia* (muy pocos años para que el test pueda dar p<α). Con ~20 años, "sin evidencia" no prueba que el tópico esté mal: puede ser falta de potencia. Los umbrales están en la sección 1.

In [ ]:
import json, re, warnings
from math import comb
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import stats
import requests

warnings.filterwarnings("ignore")
RUTA = Path(
    "."
)  # carpeta con los archivos subidos (en Colab suele ser Path("/content"))
DATA = Path("data_estandar")
RES = Path("resultados_correlaciones")
DATA.mkdir(exist_ok=True)
RES.mkdir(exist_ok=True)

YEARS = np.arange(1980, 2012)
ALPHA = 0.10  # umbral de los veredictos (series cortas, n≈20)
N_PERM, N_BOOT, SEED = 5000, 2000, 2024
MIN_AÑOS = 10  # una fuente entra al análisis si tiene al menos estos años válidos
MIN_PUNTOS = 8  # una serie con menos años no se estandariza

# Protocolo de prevalencia (del 08 del compañero). Si se sube resumen_08.json se leen de ahí.
PRINCIPAL_DESDE = 1990  # El Tiempo: primer año de la serie principal
MIN_MESES_VALIDOS = 6  # un año solo cuenta si tiene al menos estos meses válidos
RENORMALIZAR_AGENDA = (
    True  # θ se renormaliza sobre los tópicos de la agenda (sin residuales)
)
# Extensión propia para Semana en los 80 (la regla de meses válidos casi no deja años): años con pocos docs por mes
# se calculan por año completo y se marcan como BAJA confianza. Solo afecta a la fuente "semana_ext".
FALLBACK_DOCS, FALLBACK_MESES_CON_DOCS = 40, 9


def buscar(sufijo, obligatorio=True):
    hits = [
        h
        for pat in (f"*{sufijo}", f"*/*{sufijo}", f"*/*/*{sufijo}")
        for h in sorted(RUTA.glob(pat))
        if RES.name not in h.parts and DATA.name not in h.parts
    ][:1]
    if not hits and obligatorio:
        raise FileNotFoundError(
            f"No encuentro *{sufijo} bajo {RUTA.resolve()} (busco hasta 2 niveles). Sube el archivo o cambia RUTA."
        )
    return hits[0] if hits else None


f_res = buscar("resumen_08.json", obligatorio=False)
if f_res:
    cfg = json.loads(f_res.read_text(encoding="utf-8"))["config"]
    PRINCIPAL_DESDE = int(cfg.get("principal_desde", PRINCIPAL_DESDE))
    MIN_MESES_VALIDOS = int(cfg.get("min_meses_validos", MIN_MESES_VALIDOS))
    print(
        "Parámetros leídos de",
        f_res.name,
        "→ principal_desde =",
        PRINCIPAL_DESDE,
        "| min_meses_validos =",
        MIN_MESES_VALIDOS,
    )
else:
    print(
        "Sin resumen_08.json: se usan los valores por defecto (principal_desde = %d, min_meses_validos = %d)"
        % (PRINCIPAL_DESDE, MIN_MESES_VALIDOS)
    )


def zscore(s):
    s = pd.Series(s, dtype=float)
    v = s.dropna()
    if len(v) < MIN_PUNTOS or v.std(ddof=0) == 0:
        return s * np.nan
    return (s - v.mean()) / v.std(ddof=0)


CMAP = plt.get_cmap("RdBu_r").copy()
CMAP.set_bad("#e6e6e6")
rng = np.random.default_rng(SEED)

## 1. Prevalencia por macrotema y por tópico
Réplica del protocolo de `prevalencia_macro()` del 08, sin depender de `utils`/`ag`:
1. Tópicos de la agenda = los que aparecen en `macrocategorias.csv` (los que no están, p. ej. `T10`, se tratan como residuales y se excluyen).
2. θ de cada documento se renormaliza sobre los tópicos de la agenda; el macrotema suma los θ de sus tópicos.
3. Se excluyen documentos no `ok` y `es_breve`.
4. Se promedia por **mes** y luego por **año con igual peso por mes válido** (meses de `meses_validos.csv`); un año con menos de `MIN_MESES_VALIDOS` meses válidos queda en blanco.

Se calcula por fuente. `semana_ext` = Semana + años de baja confianza de los 80 (ver parámetros). Dinero tiene <10 años válidos y queda fuera.

In [ ]:
def a_bool(s):
    s = pd.Series(s)
    return (
        s
        if s.dtype == bool
        else s.astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "1.0", "t", "si", "sí"])
    )


def norm_fuente(x):
    x = str(x).lower()
    return (
        "eltiempo"
        if "tiempo" in x
        else "semana"
        if "semana" in x
        else "dinero"
        if "dinero" in x
        else x
    )


MAC = pd.read_csv(buscar("macrocategorias.csv"), keep_default_na=False)
MV = pd.read_csv(buscar("meses_validos.csv"), keep_default_na=False)
MV["fuente"] = MV["fuente"].map(norm_fuente)
MV["valido"] = a_bool(MV["valido"]).values
th = pd.read_parquet(buscar("theta_documentos.parquet"))
faltan = {"fuente", "año", "mes", "es_breve", "ok"} - set(th.columns)
assert not faltan, f"theta_documentos.parquet no tiene las columnas {faltan}"

COLS_T = sorted(c for c in th.columns if re.fullmatch(r"T\d{2}", c))
KEEP = [c for c in COLS_T if c in set(MAC["topico"])]
RESID = [c for c in COLS_T if c not in KEEP]
MACRO_ORDEN = list(dict.fromkeys(MAC["macro"]))
MACRO_DE = dict(zip(MAC["topico"], MAC["macro"]))
ETIQ = {m: m for m in MACRO_ORDEN}
ETIQ.update({t: f"{t} {n}" for t, n in zip(MAC["topico"], MAC["nombre"])})
UNIDADES = MACRO_ORDEN + KEEP
print(
    f"{len(th):,} documentos | tópicos de agenda: {len(KEEP)} | residuales (fuera de la agenda): {RESID} | macrotemas: {MACRO_ORDEN}"
)

T = th[KEEP].to_numpy(float)
masa = T.sum(1)
if RENORMALIZAR_AGENDA:
    T = np.divide(T, masa[:, None], out=np.zeros_like(T), where=masa[:, None] > 0)
D = pd.DataFrame(T, columns=KEEP)
for m in MACRO_ORDEN:
    D[m] = D[[t for t in KEEP if MACRO_DE[t] == m]].sum(axis=1)
D["fuente"] = th["fuente"].map(norm_fuente).values
D["año"] = th["año"].astype(int).values
D["mes"] = th["mes"].astype(int).values
D = D[
    (a_bool(th["ok"]).values & (masa > 0) & ~a_bool(th["es_breve"]).values)
].reset_index(drop=True)
VALIDOS = {
    f: set(
        zip(
            g.loc[g["valido"], "año"].astype(int), g.loc[g["valido"], "mes"].astype(int)
        )
    )
    for f, g in MV.groupby("fuente")
}
print("Documentos usables (ok, sin breves):", D["fuente"].value_counts().to_dict())

In [ ]:
DESDE = {"eltiempo": PRINCIPAL_DESDE}


def prevalencia_fuente(f, ext=False):
    d = D[D["fuente"] == f]
    mm = d.groupby(["año", "mes"])[UNIDADES].mean()
    mm = mm.loc[[i for i in mm.index if i in VALIDOS.get(f, set())]] if len(mm) else mm
    mm = mm[mm.index.get_level_values(0) >= DESDE.get(f, YEARS[0])]
    n_m = mm.groupby(level=0).size().reindex(YEARS, fill_value=0)
    P = mm.groupby(level=0).mean().reindex(YEARS)
    P[n_m < MIN_MESES_VALIDOS] = np.nan
    conf = pd.Series(np.where(P.notna().any(axis=1), "alta", ""), index=YEARS)
    if ext:  # años sin suficientes meses válidos: prevalencia por documento sobre el año completo, baja confianza
        for y in YEARS[P.notna().sum(axis=1) == 0]:
            dy = d[d["año"] == y]
            if (
                len(dy) >= FALLBACK_DOCS
                and dy["mes"].nunique() >= FALLBACK_MESES_CON_DOCS
            ):
                P.loc[y] = dy[UNIDADES].mean()
                conf[y] = "baja"
    return P, n_m, conf


PREV, NMES, CONF = {}, {}, {}
for nombre, (f, ext) in {
    "eltiempo": ("eltiempo", False),
    "semana": ("semana", False),
    "semana_ext": ("semana", True),
    "dinero": ("dinero", False),
}.items():
    P, n_m, conf = prevalencia_fuente(f, ext)
    anios = int(P.notna().any(axis=1).sum())
    if anios >= MIN_AÑOS:
        PREV[nombre], NMES[nombre], CONF[nombre] = P, n_m, conf
    print(
        f"{nombre:11s} años con prevalencia: {anios:2d}"
        + ("" if anios >= MIN_AÑOS else "  → fuera del análisis (< MIN_AÑOS)")
        + (
            f"  | baja confianza: {[int(y) for y in conf.index[conf == 'baja']]}"
            if ext
            else ""
        )
    )
dev = max(
    abs(P[MACRO_ORDEN].sum(axis=1, min_count=1).dropna() - 1).max()
    for P in PREV.values()
)
print(
    f"Verificación interna: los 7 macrotemas suman 1 por año (máx. desviación = {dev:.1e})"
    + ("" if RENORMALIZAR_AGENDA else " [no aplica: sin renormalizar]")
)
assert "eltiempo" in PREV, (
    "El Tiempo no tiene años válidos: revisa MIN_MESES_VALIDOS / meses_validos.csv"
)

In [ ]:
# Verificación opcional contra el 08 (si se sube prevalencias_ic.csv): suma de prevalencias por tópico dentro de cada macro
f_ic = buscar("prevalencias_ic.csv", obligatorio=False)
if f_ic:
    try:
        ref = pd.read_csv(f_ic)
        ref = ref[ref["serie"] == "eltiempo|sin_breves|mes"].copy()
        ref["macro"] = ref["topico"].str.split().str[0].map(MACRO_DE)
        ref = ref.groupby(["año", "macro"])["prev"].sum().unstack()
        P = PREV["eltiempo"][MACRO_ORDEN]
        comun = [y for y in P.dropna().index if y in ref.index]
        dif = float((P.loc[comun] - ref.loc[comun, MACRO_ORDEN]).abs().max().max())
        print(
            f"Comparación con prevalencias_ic.csv en {len(comun)} años: máx. |Δ| = {dif:.4f}",
            "✔ coincide con el 08"
            if dif < 2e-3
            else "✘ NO coincide: prueba RENORMALIZAR_AGENDA=False o ajusta MIN_MESES_VALIDOS/resumen_08.json",
        )
    except Exception as e:
        print("No se pudo comparar con prevalencias_ic.csv:", type(e).__name__, e)
else:
    print("(sin prevalencias_ic.csv: no se verifica contra el 08)")

## 2. Estándar externo adaptado a macrotemas
Mismo estándar del notebook 07, ahora al nivel de los macrotemas del compañero:

| Macrotema | Estándar |
|---|---|
| Economía | inflación, desempleo, crecimiento del PIB (Banco Mundial) |
| Conflicto y justicia | tasa de homicidios (Banco Mundial/UNODC), secuestros (CNMH, opcional), eventos congelados |
| Política e instituciones | elecciones presidenciales (calendario), eventos congelados |
| Cultura y deporte | Mundiales y Olimpiadas (calendario) |
| Sociedad, Internacional, Ciudad/territorio/tecnología | **sin estándar** (se reportan como "no evaluable") |

Cambios respecto al 07: el narcotráfico se funde en *Conflicto y justicia* (el macro agrupa conflicto armado y justicia); *desastres* se elimina (Armero 1985 y la ola invernal 2010 no tienen macro propio); los indicadores se usan **sin signo** en las correlaciones (el signo esperado se declara en la hipótesis) y con signo solo en el índice para el heatmap.

In [ ]:
CONF_MACROS = [
    "Economía",
    "Conflicto y justicia",
    "Política e instituciones",
    "Cultura y deporte",
]
assert set(CONF_MACROS) <= set(MACRO_ORDEN), (
    f"Los nombres de macro del estándar no coinciden con macrocategorias.csv: {set(CONF_MACROS) - set(MACRO_ORDEN)}"
)

# Lista congelada del notebook 06 (NO editar) y su asignación a macrotemas (principal = 1.0, secundario = 0.5)
EVENTOS = {
    1985: "Toma del Palacio de Justicia / Armero",
    1989: "Asesinato de Galán; narcoterrorismo",
    1991: "Constitución de 1991 / apertura económica",
    1993: "Muerte de Pablo Escobar",
    1996: "Proceso 8000",
    1998: "Pastrana, zona de distensión del Caguán",
    2002: "Uribe y 'seguridad democrática'",
    2006: "Reelección de Uribe / parapolítica",
    2010: "Santos / ola invernal",
}
C_, P_, E_ = "Conflicto y justicia", "Política e instituciones", "Economía"
MAPEO_EVENTOS = [
    (1985, C_, 1.0),
    (1989, C_, 1.0),
    (1989, P_, 0.5),
    (1991, P_, 1.0),
    (1991, E_, 1.0),
    (1993, C_, 1.0),
    (1996, P_, 1.0),
    (1996, C_, 1.0),
    (1998, C_, 1.0),
    (1998, P_, 0.5),
    (2002, C_, 1.0),
    (2002, P_, 0.5),
    (2006, P_, 1.0),
    (2006, C_, 0.5),
    (2010, P_, 0.5),
]
assert {a for a, _, _ in MAPEO_EVENTOS} == set(EVENTOS)
EVENTOS_MACRO = {
    m: [(a, w) for a, mm, w in MAPEO_EVENTOS if mm == m] for m in CONF_MACROS
}

# Calendario (hechos verificables). Dentro de 1990-2011 los años de Mundial y de elección presidencial COINCIDEN (≡ 2 mod 4); por eso se contrasta
# con Olimpiadas (≡ 0 mod 4) y, para fútbol, Mundial con Colombia vs sin Colombia.
CAL = {
    "elec_pres": [1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010],
    "mundial_col": [1990, 1994, 1998],
    "mundial_sin_col": [1982, 1986, 2002, 2006, 2010],
    "olimpiadas": [1980, 1984, 1988, 1992, 1996, 2000, 2004, 2008],
}
# contraste → (calendarios del grupo A, calendarios del grupo B o None = resto de años válidos, calendarios excluidos del "resto")
CONTRASTES = {
    "elec_pres vs resto": (["elec_pres"], None, []),
    "elec_pres vs olimpiadas": (["elec_pres"], ["olimpiadas"], []),
    "mundial vs olimpiadas": (["mundial_col", "mundial_sin_col"], ["olimpiadas"], []),
    "mundial_col vs mundial_sin_col": (["mundial_col"], ["mundial_sin_col"], []),
    "deporte vs resto": (["mundial_col", "olimpiadas"], None, ["mundial_sin_col"]),
}
KERNEL = {-1: 0.5, 0: 1.0, 1: 0.5}

INDICADORES = [  # signo = relación esperada con la intensidad de la agenda (hipótesis); log = conteo
    dict(
        nombre="secuestros",
        macro=C_,
        codigo=None,
        archivo="secuestros_cnmh.csv",
        signo=+1,
        log=True,
    ),
    dict(
        nombre="homicidios_tasa",
        macro=C_,
        codigo="VC.IHR.PSRC.P5",
        archivo="homicidios_tasa.csv",
        signo=+1,
        log=False,
    ),
    dict(
        nombre="inflacion",
        macro=E_,
        codigo="FP.CPI.TOTL.ZG",
        archivo="inflacion.csv",
        signo=+1,
        log=False,
    ),
    dict(
        nombre="desempleo",
        macro=E_,
        codigo="SL.UEM.TOTL.ZS",
        archivo="desempleo.csv",
        signo=+1,
        log=False,
    ),
    dict(
        nombre="crecimiento_pib",
        macro=E_,
        codigo="NY.GDP.MKTP.KD.ZG",
        archivo="crecimiento_pib.csv",
        signo=-1,
        log=False,
    ),
]


def leer_wb(codigo):
    r = requests.get(
        f"https://api.worldbank.org/v2/country/COL/indicator/{codigo}?format=json&per_page=200&date=1980:2011",
        timeout=30,
    )
    r.raise_for_status()
    j = r.json()
    if not isinstance(j, list) or len(j) < 2 or j[1] is None:
        raise ValueError("respuesta vacía")
    return pd.Series(
        {
            int(d["date"]): (np.nan if d["value"] is None else float(d["value"]))
            for d in j[1]
        },
        dtype=float,
    )


def cargar_indicador(ind):
    s_csv, s_wb, ruta = (
        pd.Series(dtype=float),
        pd.Series(dtype=float),
        DATA / ind["archivo"],
    )
    if ruta.exists():
        t = pd.read_csv(ruta)
        s_csv = pd.Series(
            pd.to_numeric(t["valor"], errors="coerce").values,
            index=t["año"].astype(int),
        ).dropna()
        s_csv = s_csv[~s_csv.index.duplicated(keep="last")]
    if ind["codigo"]:
        try:
            s_wb = leer_wb(ind["codigo"]).dropna()
        except Exception as e:
            print(
                f"  ⚠ {ind['nombre']}: sin descarga del Banco Mundial ({type(e).__name__}); solo CSV manual"
            )
    elif not ruta.exists():
        print(
            f"  ⚠ {ind['nombre']}: opcional, coloca {ruta} (columnas año,valor) para usarlo"
        )
    return s_csv.combine_first(s_wb).reindex(YEARS)


SERIES, RESUMEN = {}, []
for ind in INDICADORES:
    s = cargar_indicador(ind)
    SERIES[ind["nombre"]] = np.log1p(s) if ind["log"] else s
    v = s.dropna()
    RESUMEN.append(
        dict(
            indicador=ind["nombre"],
            macro=ind["macro"],
            años=len(v),
            desde=int(v.index.min()) if len(v) else None,
            hasta=int(v.index.max()) if len(v) else None,
            usable=len(v) >= MIN_PUNTOS,
        )
    )
SERIES = pd.DataFrame(SERIES)
SERIES.index.name = "año"
print(pd.DataFrame(RESUMEN).to_string(index=False))
SERIES_USABLES = [r["indicador"] for r in RESUMEN if r["usable"]]
SIGNO = {i["nombre"]: i["signo"] for i in INDICADORES}

In [ ]:
# Índice del estándar por macrotema (solo para el heatmap comparativo; las correlaciones usan las series sin signo)
B_EN_COMBINADO = {m: True for m in CONF_MACROS}
B_EN_COMBINADO[E_] = False  # economía: un solo evento (1991) dominaría la fila


def serie_B(m):
    s = pd.Series(0.0, index=YEARS)
    for a, w in EVENTOS_MACRO[m]:
        for lag, k in KERNEL.items():
            if (a + lag) in s.index:
                s[a + lag] += w * k
    if m == P_:
        s[[a for a in CAL["elec_pres"] if a in s.index]] += 1.0
    if m == "Cultura y deporte":
        for nom, w in (
            ("mundial_col", 1.0),
            ("mundial_sin_col", 0.5),
            ("olimpiadas", 0.5),
        ):
            s[[a for a in CAL[nom] if a in s.index]] += w
    return s if s.any() else None


STD_IDX = pd.DataFrame(np.nan, index=MACRO_ORDEN, columns=YEARS)
for m in CONF_MACROS:
    cols = [
        i["nombre"]
        for i in INDICADORES
        if i["macro"] == m and i["nombre"] in SERIES_USABLES
    ]
    a = (
        zscore(
            pd.concat([zscore(SERIES[c]) * SIGNO[c] for c in cols], axis=1).mean(
                axis=1, skipna=True
            )
        )
        if cols
        else None
    )
    b = serie_B(m)
    b = zscore(b) if b is not None else None
    partes = [p for p in (a, b if B_EN_COMBINADO[m] else None) if p is not None]
    if partes:
        STD_IDX.loc[m] = (
            zscore(pd.concat(partes, axis=1).mean(axis=1, skipna=False))
            if len(partes) > 1
            else partes[0]
        ).values
print(
    "Años con estándar por macro:", STD_IDX.notna().sum(axis=1)[CONF_MACROS].to_dict()
)

## 3. Hipótesis congeladas
Cada fila dice qué unidad (macrotema o tópico) debería moverse con qué componente del estándar y en qué sentido (`esp` = +1 / −1). **Se congelan antes de mirar los resultados.** Las hipótesis a nivel de tópico son más nítidas que las de macrotema (p. ej. *Elecciones y partidos* ↔ años electorales) y sirven de control: si un tópico con nombre inequívoco no responde a su evento, es mala señal.

Pruebas por tipo:
- **serie** (indicador anual): ρ de Spearman en **primeras diferencias** (lag 0; test de permutación unilateral, IC por bootstrap, concordancia de signo) y en **niveles** (null por desplazamiento circular, que respeta la autocorrelación). También se reportan lag ±1.
- **evento**: cambio de la prevalencia (z) en el año del evento frente a los 3 años previos, promedio ponderado por peso del evento, contra años elegidos al azar.
- **calendario**: diferencia de la prevalencia (z) entre años del grupo A y del grupo B, test de permutación exacto por remuestreo.

In [ ]:
def H(id, unidad, tipo, comp, esp=+1):
    return dict(id=id, unidad=unidad, tipo=tipo, comp=comp, esp=esp)


HIP = [
    H("E1", "Economía", "serie", "inflacion"),
    H("E2", "Economía", "serie", "desempleo"),
    H("E3", "Economía", "serie", "crecimiento_pib", -1),
    H("C1", "Conflicto y justicia", "serie", "homicidios_tasa"),
    H("C2", "Conflicto y justicia", "serie", "secuestros"),
    H("C3", "Conflicto y justicia", "evento", "eventos"),
    H("P1", "Política e instituciones", "calendario", "elec_pres vs resto"),
    H("P2", "Política e instituciones", "evento", "eventos"),
    H("D1", "Cultura y deporte", "calendario", "deporte vs resto"),
    H("T1", "T13", "calendario", "elec_pres vs resto"),
    H("T2", "T13", "calendario", "elec_pres vs olimpiadas"),
    H("T3", "T15", "calendario", "mundial vs olimpiadas"),
    H("T4", "T15", "calendario", "mundial_col vs mundial_sin_col"),
    H("T5", "T06", "serie", "inflacion"),
    H("T6", "T06", "serie", "desempleo"),
    H("T7", "T06", "serie", "crecimiento_pib", -1),
    H("T8", "T02", "serie", "homicidios_tasa"),
]
assert all(h["unidad"] in UNIDADES for h in HIP), (
    "Alguna unidad de las hipótesis no existe en macrocategorias.csv"
)
pd.DataFrame([{**h, "unidad": ETIQ[h["unidad"]]} for h in HIP]).to_csv(
    RES / "hipotesis_congeladas.csv", index=False
)
print(
    pd.DataFrame([{**h, "unidad": ETIQ[h["unidad"]]} for h in HIP]).to_string(
        index=False
    )
)

## 4. Funciones de prueba

In [ ]:
def _corr_rows(ra, RB):
    a = (ra - ra.mean()) / ra.std()
    B = (RB - RB.mean(1, keepdims=True)) / RB.std(1, keepdims=True)
    return (B @ a) / len(ra)


def spearman_perm(a, b, sgn):
    n = len(a)
    if n < 6 or np.std(a) == 0 or np.std(b) == 0:
        return np.nan, np.nan
    ra, rb = stats.rankdata(a), stats.rankdata(b)
    obs = float(np.corrcoef(ra, rb)[0, 1])
    nulls = _corr_rows(ra, rb[np.argsort(rng.random((N_PERM, n)), axis=1)])
    return obs, (1 + np.sum(nulls * sgn >= obs * sgn)) / (N_PERM + 1)


def spearman_circ(
    a, b, sgn
):  # null por desplazamiento circular (conserva la autocorrelación de cada serie)
    n = len(a)
    ra, rb = stats.rankdata(a), stats.rankdata(b)
    if n < MIN_PUNTOS or np.std(a) == 0 or np.std(b) == 0:
        return np.nan, np.nan
    obs = float(np.corrcoef(ra, rb)[0, 1])
    nulls = np.array([np.corrcoef(ra, np.roll(rb, k))[0, 1] for k in range(1, n)])
    return obs, (1 + np.sum(nulls * sgn >= obs * sgn)) / n


def spearman_boot(a, b):
    n = len(a)
    idx = rng.integers(0, n, (N_BOOT, n))
    RA, RB = stats.rankdata(a[idx], axis=1), stats.rankdata(b[idx], axis=1)
    RA, RB = RA - RA.mean(1, keepdims=True), RB - RB.mean(1, keepdims=True)
    den = np.sqrt((RA**2).sum(1) * (RB**2).sum(1))
    r = np.where(den > 0, (RA * RB).sum(1) / np.where(den > 0, den, 1), np.nan)
    return tuple(np.nanpercentile(r, [2.5, 97.5]))


def rho_simple(p, x):
    d = pd.concat([p, x], axis=1, keys=["p", "x"]).dropna()
    return (
        float(stats.spearmanr(d["p"], d["x"])[0])
        if len(d) >= 6 and d["p"].std() > 0 and d["x"].std() > 0
        else np.nan
    ), len(d)


def metricas_serie(p, x, s):
    p, x = p.reindex(YEARS), x.reindex(YEARS)
    r = {}
    d = pd.concat([p, x], axis=1, keys=["p", "x"]).dropna()
    r["n_nivel"] = len(d)
    r["rho_nivel"], r["p_nivel"] = (
        spearman_circ(d["p"].values, d["x"].values, s)
        if len(d) >= MIN_PUNTOS
        else (np.nan, np.nan)
    )
    dp, dx = p.diff(), x.diff()
    dd = pd.concat([dp, dx], axis=1, keys=["p", "x"]).dropna()
    r["n_dif"] = len(dd)
    if len(dd) >= 8:
        r["rho_dif0"], r["p_dif"] = spearman_perm(dd["p"].values, dd["x"].values, s)
        r["ic_lo"], r["ic_hi"] = spearman_boot(dd["p"].values, dd["x"].values)
        m = (dd["p"].values != 0) & (dd["x"].values != 0)
        acuerdo = (np.sign(dd["p"].values) * np.sign(dd["x"].values) * s)[m] > 0
        r["concordancia"] = float(acuerdo.mean()) if m.sum() else np.nan
        r["p_concord"] = (
            stats.binomtest(
                int(acuerdo.sum()), int(m.sum()), 0.5, alternative="greater"
            ).pvalue
            if m.sum()
            else np.nan
        )
    else:
        r.update(
            rho_dif0=np.nan,
            p_dif=np.nan,
            ic_lo=np.nan,
            ic_hi=np.nan,
            concordancia=np.nan,
            p_concord=np.nan,
        )
    r["rho_dif_lag1"] = rho_simple(dp, dx.shift(1))[
        0
    ]  # la agenda reacciona un año después
    r["rho_dif_lead1"] = rho_simple(dp, dx.shift(-1))[0]  # la agenda se anticipa un año
    return r


def evento_estudio(z, eventos):
    z = z.reindex(YEARS)
    base = lambda t: z.reindex([t - 3, t - 2, t - 1]).dropna()
    elegibles = [t for t in YEARS if np.isfinite(z[t]) and len(base(t)) >= 2]
    delta = {t: z[t] - base(t).mean() for t in elegibles}
    ev = [(t, w) for t, w in eventos if t in delta]
    if len(ev) < 3:
        return dict(
            n_eventos=len(ev),
            delta0=np.nan,
            delta_lag1=np.nan,
            p=np.nan,
            potencia=False,
        )
    ts, ws = np.array([t for t, _ in ev]), np.array([w for _, w in ev])
    obs = float(np.sum(ws * np.array([delta[t] for t in ts])) / ws.sum())
    dl = np.array([delta[t] for t in elegibles])
    k = len(ev)
    nulls = np.array(
        [
            np.sum(rng.permutation(ws) * dl[rng.choice(len(dl), k, replace=False)])
            / ws.sum()
            for _ in range(N_PERM)
        ]
    )
    d1 = [z.get(t + 1, np.nan) - base(t).mean() for t in ts]
    return dict(
        n_eventos=k,
        delta0=obs,
        delta_lag1=float(np.nanmean(d1)) if np.isfinite(d1).any() else np.nan,
        p=(1 + np.sum(nulls >= obs)) / (N_PERM + 1),
        potencia=True,
    )


def grupos_calendario(z, nombre):
    A_n, B_n, excl_n = CONTRASTES[nombre]
    ok = set(int(t) for t in z.index[z.notna()])
    A = sorted({a for n in A_n for a in CAL[n]} & ok)
    excl = {a for n in excl_n for a in CAL[n]}
    B = sorted(
        ({a for n in B_n for a in CAL[n]} if B_n else ok - set(A) - excl) & ok - set(A)
    )
    return A, B


def calendario_efecto(z, nombre, s=+1):
    z = z.reindex(YEARS)
    A, B = grupos_calendario(z, nombre)
    if len(A) < 2 or len(B) < 2:
        return dict(n_A=len(A), n_B=len(B), efecto=np.nan, p=np.nan, potencia=False)
    obs = float(z[A].mean() - z[B].mean())
    pool = z[A + B].values
    k = len(A)
    nulls = np.array(
        [
            pool[(pi := rng.permutation(len(pool)))[:k]].mean() - pool[pi[k:]].mean()
            for _ in range(N_PERM)
        ]
    )
    return dict(
        n_A=len(A),
        n_B=len(B),
        efecto=obs,
        p=(1 + np.sum(nulls * s >= obs * s)) / (N_PERM + 1),
        potencia=(1 / comb(len(A) + len(B), len(A))) < ALPHA,
    )


def bh(p):
    p = np.asarray(p, float)
    q = np.full(len(p), np.nan)
    ok = np.isfinite(p)
    n = ok.sum()
    if n == 0:
        return q
    o = np.argsort(p[ok])
    v = p[ok][o] * n / (np.arange(n) + 1)
    v = np.minimum.accumulate(v[::-1])[::-1]
    tmp = np.empty(n)
    tmp[o] = np.minimum(v, 1)
    q[ok] = tmp
    return q

## 5. Correlaciones: todas las unidades × todo el estándar, y las hipótesis
Cuidado con la lectura: con α = 0.10, alrededor del 10 % de pares al azar saldrían como *Soporta* (y otro 10 % como *Contradice*). Por eso se reportan también `q_bh` (corrección por pruebas múltiples) y `rank_especificidad` (posición del tópico esperado entre todos los de su nivel: 1/17 es mucho más convincente que 6/17). Un *Soporta* con rank 1 y q bajo es evidencia sólida.

Primero la **matriz completa exploratoria** (cada unidad contra cada componente; sirve para ver especificidad: ¿es el macrotema esperado el que más se mueve con el indicador?). Luego las **hipótesis congeladas** con test, IC y veredicto.

In [ ]:
Z_PREV = {f: P.apply(zscore) for f, P in PREV.items()}
MAT_SERIE, MAT_CAL = {}, {}
for f, P in PREV.items():
    MAT_SERIE[f] = pd.DataFrame(
        {
            c: {
                u: rho_simple(P[u].reindex(YEARS).diff(), SERIES[c].diff())[0]
                for u in UNIDADES
            }
            for c in SERIES_USABLES
        }
    )
    MAT_CAL[f] = pd.DataFrame(
        {
            k: {
                u: (
                    lambda A, B: (
                        (Z_PREV[f][u][A].mean() - Z_PREV[f][u][B].mean())
                        if len(A) >= 2 and len(B) >= 2
                        else np.nan
                    )
                )(*grupos_calendario(Z_PREV[f][u], k))
                for u in UNIDADES
            }
            for k in CONTRASTES
        }
    )
print("Matrices calculadas para:", list(PREV))


def rank_especificidad(f, h, valor):
    if not np.isfinite(valor):
        return ""
    M = MAT_SERIE[f] if h["tipo"] == "serie" else MAT_CAL[f]
    col = h["comp"]
    if col not in M.columns:
        return ""
    pares = MACRO_ORDEN if h["unidad"] in MACRO_ORDEN else KEEP
    v = (M.loc[pares, col] * h["esp"]).dropna()
    me = valor * h["esp"]
    return f"{int((v > me).sum()) + 1}/{len(v)}"


filas = []
for f, P in PREV.items():
    for h in HIP:
        z = Z_PREV[f][h["unidad"]]
        r = dict(
            fuente=f,
            id=h["id"],
            unidad=ETIQ[h["unidad"]],
            tipo=h["tipo"],
            comp=h["comp"],
            esp=h["esp"],
        )
        if h["tipo"] == "serie":
            if h["comp"] not in SERIES_USABLES:
                r["veredicto"] = "Sin datos del estándar"
                filas.append(r)
                continue
            r.update(metricas_serie(P[h["unidad"]], SERIES[h["comp"]], h["esp"]))
            r["rank_especificidad"] = rank_especificidad(f, h, r["rho_dif0"])
            r["p_primario"] = r["p_dif"]
        elif h["tipo"] == "evento":
            ev = evento_estudio(z, EVENTOS_MACRO[h["unidad"]])
            r.update(ev)
            r["efecto"] = ev["delta0"]
            r["p_primario"] = ev["p"]
        else:
            ca = calendario_efecto(z, h["comp"], h["esp"])
            r.update(ca)
            r["p_primario"] = ca["p"]
            r["rank_especificidad"] = rank_especificidad(f, h, ca["efecto"])
        filas.append(r)
R = pd.DataFrame(filas)
R["q_bh"] = np.nan
for f in PREV:
    m = R["fuente"] == f
    R.loc[m, "q_bh"] = bh(R.loc[m, "p_primario"].values)


def veredicto(r):
    if r.get("veredicto") == "Sin datos del estándar":
        return r["veredicto"]
    s = r["esp"]
    if r["tipo"] == "serie":
        if not np.isfinite(r.get("rho_dif0", np.nan)) and not np.isfinite(
            r.get("rho_nivel", np.nan)
        ):
            return "Sin potencia"
        ap = [
            np.isfinite(r["p_dif"]) and r["rho_dif0"] * s > 0 and r["p_dif"] < ALPHA,
            np.isfinite(r["p_nivel"])
            and r["rho_nivel"] * s > 0
            and r["p_nivel"] < ALPHA,
        ]
        co = [
            np.isfinite(r["p_dif"]) and r["p_dif"] > 1 - ALPHA,
            np.isfinite(r["p_nivel"]) and r["p_nivel"] > 1 - ALPHA,
        ]
        if any(ap) and any(co):
            return "Mixto"
        if any(ap):
            return "Soporta (fuerte)" if all(ap) else "Soporta (parcial)"
        return "Contradice" if any(co) else "Sin evidencia"
    if not r.get("potencia", False):
        return "Sin potencia"
    return (
        "Soporta"
        if r["p"] < ALPHA
        else ("Contradice" if r["p"] > 1 - ALPHA else "Sin evidencia")
    )


R["veredicto"] = R.apply(veredicto, axis=1)
cols = [
    "fuente",
    "id",
    "unidad",
    "tipo",
    "comp",
    "esp",
    "n_dif",
    "rho_dif0",
    "ic_lo",
    "ic_hi",
    "p_dif",
    "concordancia",
    "p_concord",
    "rho_dif_lag1",
    "rho_dif_lead1",
    "n_nivel",
    "rho_nivel",
    "p_nivel",
    "n_eventos",
    "delta0",
    "delta_lag1",
    "n_A",
    "n_B",
    "efecto",
    "p",
    "rank_especificidad",
    "q_bh",
    "veredicto",
]
R = R.reindex(columns=cols)
R.to_csv(RES / "resultados_hipotesis.csv", index=False)
pd.set_option("display.width", 250, "display.max_columns", 40)
for f in PREV:
    print(f"\n=== {f} ===")
    print(
        R[R["fuente"] == f][
            [
                "id",
                "unidad",
                "tipo",
                "comp",
                "rho_dif0",
                "p_dif",
                "rho_nivel",
                "p_nivel",
                "delta0",
                "efecto",
                "p",
                "n_A",
                "n_B",
                "rank_especificidad",
                "veredicto",
            ]
        ]
        .round(3)
        .to_string(index=False)
    )

## 6. Qué unidades se validan, cuáles se documentan y descartan
Estado por unidad con la **fuente principal (El Tiempo)**; Semana se muestra al lado como contraste independiente.
- **Validado**: soporta al menos la mitad de las hipótesis evaluables.
- **Parcial**: soporta alguna, pero menos de la mitad.
- **Sin respaldo → documentar y descartar**: hay hipótesis evaluables y ninguna soporta (con la salvedad de potencia).
- **No concluyente**: todas las hipótesis son *Sin potencia* / sin datos.
- **No evaluable (sin estándar)**: la unidad no tiene hipótesis.

In [ ]:
def estado(sub):
    ev = sub[~sub["veredicto"].isin(["Sin potencia", "Sin datos del estándar"])]
    if len(ev) == 0:
        return "No concluyente"
    sop = ev["veredicto"].str.startswith("Soporta").sum()
    if sop == 0:
        return "Sin respaldo → documentar y descartar"
    return "Validado" if sop >= len(ev) / 2 else "Parcial"


filas = []
for u in UNIDADES:
    hu = R[R["unidad"] == ETIQ[u]]
    fila = dict(
        unidad=ETIQ[u],
        nivel="macro" if u in MACRO_ORDEN else "tópico",
        hipótesis=sorted(set(hu["id"])),
    )
    for f in PREV:
        s = hu[hu["fuente"] == f]
        fila[f"estado_{f}"] = estado(s) if len(s) else "No evaluable (sin estándar)"
        fila[f"soporta_{f}"] = (
            f"{int(s['veredicto'].str.startswith('Soporta').sum())}/{len(s)}"
            if len(s)
            else ""
        )
    filas.append(fila)
ESTADO = pd.DataFrame(filas)
ESTADO.to_csv(RES / "estado_unidades.csv", index=False)
print(
    ESTADO[ESTADO["hipótesis"].map(len) > 0]
    .drop(columns="hipótesis")
    .to_string(index=False)
)
print(
    "\nSin estándar (no evaluables):",
    [ETIQ[u] for u in UNIDADES if not (R["unidad"] == ETIQ[u]).any()][:8],
    "...",
)

## 7. Gráficas
**F1** prevalencia por macrotema (z por fila, mismo estilo que el 06/07) frente al índice del estándar. **F2** matrices completas (ρ en diferencias y efecto de calendario), con las hipótesis recuadradas. **F3** resumen de hipótesis con IC. **F4** series superpuestas.

In [ ]:
def plot_heatmap(ax, M, titulo, yticks=True, vmax=2.5):
    im = ax.imshow(
        np.ma.masked_invalid(M.values.astype(float)),
        aspect="auto",
        cmap=CMAP,
        vmin=-vmax,
        vmax=vmax,
    )
    ax.set_xticks(range(0, len(YEARS), 4))
    ax.set_xticklabels(YEARS[::4], rotation=45)
    ax.set_yticks(range(len(M.index)) if yticks else [])
    if yticks:
        ax.set_yticklabels([ETIQ.get(i, i) for i in M.index], fontsize=8)
    ax.set_title(titulo, fontsize=10)
    return im


paneles = [
    (f, "Prevalencia · " + f) for f in ("eltiempo", "semana_ext") if f in PREV
] + [(None, "Estándar (índice por macrotema)")]
fig, axs = plt.subplots(1, len(paneles), figsize=(6.2 * len(paneles), 4.4))
for i, (ax, (f, t)) in enumerate(zip(np.atleast_1d(axs), paneles)):
    im = plot_heatmap(
        ax, (Z_PREV[f][MACRO_ORDEN].T if f else STD_IDX), t, yticks=(i == 0)
    )
fig.colorbar(im, ax=axs, shrink=0.85, label="z-score por fila")
fig.suptitle(
    "F1 · Agenda por macrotema vs estándar, 1980–2011 (gris = sin dato / baja cobertura)",
    y=1.03,
)
fig.savefig(RES / "F1_prevalencia_vs_estandar.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
def plot_matriz(f):
    ms, mc = MAT_SERIE[f], MAT_CAL[f]
    fig, axs = plt.subplots(
        1,
        2,
        figsize=(13, 9.5),
        gridspec_kw={"width_ratios": [ms.shape[1] + 1.5, mc.shape[1] + 1.5]},
    )
    for j, (ax, M, vmax, tit) in enumerate(
        (
            (axs[0], ms, 1.0, "ρ de Spearman, primeras diferencias (lag 0)"),
            (axs[1], mc, 2.5, "Efecto de calendario: z medio (años A − años B)"),
        )
    ):
        im = ax.imshow(
            np.ma.masked_invalid(M.values.astype(float)),
            cmap=CMAP,
            vmin=-vmax,
            vmax=vmax,
            aspect="auto",
        )
        ax.set_xticks(range(M.shape[1]))
        ax.set_xticklabels(M.columns, rotation=40, ha="right", fontsize=8)
        ax.set_yticks(range(M.shape[0]))
        ax.set_yticklabels([ETIQ[u] for u in M.index] if j == 0 else [], fontsize=8)
        for i in range(M.shape[0]):
            for k in range(M.shape[1]):
                v = M.values[i, k]
                if np.isfinite(v):
                    ax.text(k, i, f"{v:.2f}", ha="center", va="center", fontsize=6.5)
        ax.axhline(len(MACRO_ORDEN) - 0.5, color="k", lw=1.2)
        ax.set_title(tit, fontsize=10)
        fig.colorbar(im, ax=ax, shrink=0.6)
        for h in HIP:
            if (
                h["tipo"] == ("serie" if j == 0 else "calendario")
                and h["comp"] in M.columns
                and h["unidad"] in M.index
            ):
                ax.add_patch(
                    Rectangle(
                        (
                            list(M.columns).index(h["comp"]) - 0.5,
                            list(M.index).index(h["unidad"]) - 0.5,
                        ),
                        1,
                        1,
                        fill=False,
                        ec="k",
                        lw=2,
                    )
                )
    fig.suptitle(
        f"F2 · {f}: unidades × estándar (recuadro = hipótesis congelada; línea = separa macrotemas de tópicos)",
        y=0.93,
    )
    fig.savefig(RES / f"F2_matriz_{f}.png", dpi=150, bbox_inches="tight")
    plt.show()


for f in [f for f in ("eltiempo", "semana_ext") if f in PREV]:
    plot_matriz(f)

In [ ]:
colores = {"eltiempo": "tab:blue", "semana": "tab:orange", "semana_ext": "tab:green"}
fig, axs = plt.subplots(1, 2, figsize=(14, 6.5), gridspec_kw={"width_ratios": [1.1, 1]})
for ax, tipos, xl in (
    (axs[0], ["serie"], "ρ de Spearman en primeras diferencias (IC 95 % bootstrap)"),
    (axs[1], ["evento", "calendario"], "cambio / efecto en z (evento o calendario)"),
):
    sub = R[R["tipo"].isin(tipos) & ~R["veredicto"].isin(["Sin datos del estándar"])]
    ids = list(dict.fromkeys(sub["id"]))
    for k, i in enumerate(ids):
        for jf, f in enumerate([f for f in PREV if f in set(sub["fuente"])]):
            r = sub[(sub["id"] == i) & (sub["fuente"] == f)]
            if r.empty:
                continue
            r = r.iloc[0]
            y = k + (jf - 0.5 * (len(PREV) - 1)) * 0.22
            v = (
                r["rho_dif0"]
                if tipos == ["serie"]
                else (r["delta0"] if r["tipo"] == "evento" else r["efecto"])
            )
            if not np.isfinite(v):
                continue
            if tipos == ["serie"] and np.isfinite(r["ic_lo"]):
                ax.plot([r["ic_lo"], r["ic_hi"]], [y, y], color=colores[f], lw=1.2)
            ax.plot(
                v,
                y,
                "o",
                color=colores[f],
                mfc=colores[f]
                if str(r["veredicto"]).startswith("Soporta")
                else "white",
                ms=7,
            )
    ax.axvline(0, color="gray", lw=0.8)
    ax.set_yticks(range(len(ids)))
    ax.set_yticklabels(
        [
            f"{i} · {sub[sub['id'] == i]['unidad'].iloc[0][:26]} ~ {sub[sub['id'] == i]['comp'].iloc[0][:22]}"
            for i in ids
        ],
        fontsize=7,
    )
    ax.invert_yaxis()
    ax.set_xlabel(xl, fontsize=8)
for f in PREV:
    axs[0].plot([], [], "o", color=colores[f], label=f)
axs[0].legend(fontsize=7, title="relleno = Soporta", title_fontsize=7, loc="upper left")
fig.suptitle("F3 · Hipótesis congeladas: efecto observado por fuente", y=0.99)
fig.tight_layout()
fig.savefig(RES / "F3_hipotesis.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
hs = [h for h in HIP if h["tipo"] == "serie" and h["comp"] in SERIES_USABLES]
f0 = "eltiempo"
if hs:
    nc = 3
    nr = int(np.ceil(len(hs) / nc))
    fig, axs = plt.subplots(nr, nc, figsize=(15, 3.2 * nr), sharex=True)
    axs = np.atleast_1d(axs).ravel()
    for ax, h in zip(axs, hs):
        r = R[(R["fuente"] == f0) & (R["id"] == h["id"])].iloc[0]
        ax.plot(
            YEARS,
            Z_PREV[f0][h["unidad"]].values,
            color="tab:blue",
            marker="o",
            ms=3,
            label="prevalencia (z)",
        )
        ax.plot(
            YEARS,
            (zscore(SERIES[h["comp"]]) * h["esp"]).values,
            color="gray",
            marker="s",
            ms=2.5,
            label=f"{h['comp']} ({'+' if h['esp'] > 0 else '−'}) (z)",
        )
        ax.axhline(0, color="k", lw=0.4)
        ax.set_title(
            f"{h['id']} · {ETIQ[h['unidad']]}\nρΔ={r['rho_dif0']:.2f}  ρnivel={r['rho_nivel']:.2f}  → {r['veredicto']}",
            fontsize=8,
        )
        ax.legend(fontsize=6, loc="upper left")
    for ax in axs[len(hs) :]:
        ax.axis("off")
    fig.suptitle(
        "F4 · El Tiempo: prevalencia vs indicador (con el signo esperado: las curvas deberían moverse juntas)",
        y=1.0,
    )
    fig.tight_layout()
    fig.savefig(RES / "F4_series_superpuestas.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Exportar

In [ ]:
for f, P in PREV.items():
    P.round(5).to_csv(RES / f"prevalencia_{f}.csv")
    pd.DataFrame({"meses_validos": NMES[f], "confianza": CONF[f]}).to_csv(
        RES / f"cobertura_{f}.csv"
    )
    MAT_SERIE[f].round(4).to_csv(RES / f"matriz_rho_dif_{f}.csv")
    MAT_CAL[f].round(4).to_csv(RES / f"matriz_calendario_{f}.csv")
STD_IDX.round(4).to_csv(RES / "estandar_indice_macro.csv")
SERIES.round(4).to_csv(RES / "estandar_series.csv")
print(sorted(p.name for p in RES.iterdir()))